# ARI711S – Artificial Intelligence Group Project 2026

**Qualification:** Bachelor of Computer Science (Software Development)  
**Course:** Artificial Intelligence – ARI711S  
**Assessment:** Group Project  
**Due Date:** 30 April 2026

---

## Group Members
- Tangi
- Matias
- Aguero
- Meke
- Trizzy

---

## Table of Contents
1. [Part 1: Search Algorithms – Flight Connections Between Cities](#part1)
2. [Part 2: Optimization – Hospital Shift Scheduler](#part2)
3. [Part 3: Machine Learning – Traffic Sign Recognition](#part3)

---
# Part 1: Search Algorithms – Flight Connections Between Cities
<a id='part1'></a>

## Background

This part solves the problem of finding the **shortest flight path** between two cities using a graph search algorithm. The cities and direct flight routes are loaded from CSV files, and the program uses **Breadth-First Search (BFS)** to guarantee the minimum number of connections.

## Approach

We modelled the problem as a **graph search problem**:
- Each **city** is a node in the graph
- Each **direct flight route** is an edge between two city nodes
- The goal is to find the shortest path (fewest edges) from a source city to a target city

BFS was chosen over DFS because BFS explores cities level by level, guaranteeing the first solution found uses the fewest connections.

## Data Structures
- `city_name_to_ids` – maps city names to their IDs
- `cities` – maps each city ID to its details and outgoing flights
- `airlines` – maps airline IDs to airline names
- `flight_airline_map` – maps flight IDs to airline IDs

In [ ]:
# Part 1 – Flight Connections Search
# ARI711S Artificial Intelligence – Group Project 2026

import csv
from collections import deque

# ── Data stores ──────────────────────────────────────────
city_name_to_ids = {}   # city name (lowercase) -> set of city_ids
cities = {}             # city_id -> {name, country, flights}
airlines = {}           # airline_id -> airline_name
flight_airline_map = {} # flight_id -> airline_id


# ── Node and Frontier classes ─────────────────────────────

class Node:
    def __init__(self, state, parent, action):
        self.state = state    # city_id
        self.parent = parent  # parent Node
        self.action = action  # (flight_id, city_id) that led here


class StackFrontier:
    """LIFO frontier – Depth-First Search"""
    def __init__(self):
        self._frontier = []

    def add(self, node):
        self._frontier.append(node)

    def contains_state(self, state):
        return any(node.state == state for node in self._frontier)

    def empty(self):
        return len(self._frontier) == 0

    def remove(self):
        if self.empty():
            raise Exception("Frontier is empty.")
        return self._frontier.pop()


class QueueFrontier(StackFrontier):
    """FIFO frontier – Breadth-First Search (guarantees shortest path)"""
    def remove(self):
        if self.empty():
            raise Exception("Frontier is empty.")
        return self._frontier.pop(0)


# ── Data loading ──────────────────────────────────────────

def load_data(directory):
    with open(f"{directory}/airlines.csv", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            airlines[row["airline_id"]] = row["airline_name"]

    with open(f"{directory}/cities.csv", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            cid = row["city_id"]
            cities[cid] = {"name": row["city_name"], "country": row["country"], "flights": set()}
            city_name_to_ids.setdefault(row["city_name"].lower(), set()).add(cid)

    with open(f"{directory}/flights.csv", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            src, dst, fid, aid = row["source_city_id"], row["destination_city_id"], row["flight_id"], row["airline_id"]
            if src in cities:
                cities[src]["flights"].add((fid, dst))
            flight_airline_map[fid] = aid


# ── Required functions ────────────────────────────────────

def neighbors_for_city(city_id):
    """Returns set of (flight_id, city_id) for all direct flights from city_id."""
    if city_id not in cities:
        return set()
    return cities[city_id]["flights"]


def shortest_path(source, target):
    """
    Returns the shortest path from source to target as a list of
    (flight_id, city_id) tuples, or None if no path exists.
    Uses BFS to guarantee minimum number of connections.
    """
    if source == target:
        return []

    frontier = QueueFrontier()
    frontier.add(Node(state=source, parent=None, action=None))
    explored = set()

    while not frontier.empty():
        node = frontier.remove()
        if node.state in explored:
            continue
        explored.add(node.state)

        for flight_id, neighbor_id in neighbors_for_city(node.state):
            if neighbor_id in explored:
                continue
            child = Node(state=neighbor_id, parent=node, action=(flight_id, neighbor_id))
            # check goal on add for efficiency
            if neighbor_id == target:
                return _reconstruct_path(child)
            if not frontier.contains_state(neighbor_id):
                frontier.add(child)

    return None  # no path exists


def _reconstruct_path(node):
    path = []
    while node.action is not None:
        path.append(node.action)
        node = node.parent
    path.reverse()
    return path


print("Part 1 code loaded successfully.")

## Loading the Dataset

We load three CSV files:
- `cities.csv` – city IDs, names, and countries
- `flights.csv` – direct routes between cities with airline info
- `airlines.csv` – airline names

In [ ]:
# Load the dataset
# Make sure the 'data' folder with cities.csv, flights.csv, airlines.csv is in part1/
load_data("part1/data")
print(f"Loaded {len(cities)} cities and {len(airlines)} airlines.")
print(f"Total flight routes: {sum(len(c['flights']) for c in cities.values())}")

## Test Cases

In [ ]:
# Test 1: Windhoek to Cairo (expected: 3 connections)
src = next(iter(city_name_to_ids["windhoek"]))
tgt = next(iter(city_name_to_ids["cairo"]))
path = shortest_path(src, tgt)

print(f"{len(path)} flight connection(s) from Windhoek to Cairo:\n")
current = src
for i, (fid, dest_id) in enumerate(path, 1):
    src_name = cities[current]["name"]
    dst_name = cities[dest_id]["name"]
    airline = airlines.get(flight_airline_map.get(fid, ""), "Unknown")
    print(f"  {i}: {src_name} to {dst_name} (Flight {fid}, {airline})")
    current = dest_id

In [ ]:
# Test 2: Same city should return empty list
cid = next(iter(city_name_to_ids["windhoek"]))
result = shortest_path(cid, cid)
print(f"Same city test: {result}")  # expected: []

# Test 3: Direct flight (1 connection)
src = next(iter(city_name_to_ids["windhoek"]))
tgt = next(iter(city_name_to_ids["lusaka"]))
path = shortest_path(src, tgt)
print(f"Windhoek to Lusaka: {len(path)} connection(s)")  # expected: 1

# Test 4: No path (isolated city)
cities["999"] = {"name": "Isolated", "country": "Nowhere", "flights": set()}
result = shortest_path(src, "999")
print(f"No path test: {result}")  # expected: None
del cities["999"]  # clean up

# Test 5: neighbors_for_city
jnb = next(iter(city_name_to_ids["johannesburg"]))
neighbors = neighbors_for_city(jnb)
print(f"Johannesburg has {len(neighbors)} direct routes")

---
# Part 2: Optimization – Hospital Shift Scheduler
<a id='part2'></a>

## Background

Nurse scheduling is a **Constraint Satisfaction Problem (CSP)**. We need to assign nurses to 21 weekly shifts (7 days × 3 shifts per day) while satisfying all constraints.

## Problem Formulation

| CSP Component | Our Definition |
|---|---|
| **Variables** | 21 shift slots (e.g. Monday_Morning, Monday_Afternoon, ..., Sunday_Night) |
| **Domain** | All available nurses for each shift |
| **Unary constraint** | A nurse cannot work on a day they have pre-approved leave |
| **Binary constraint** | A nurse on Night shift cannot work the next Morning (rest period) |
| **Higher-order constraint** | No nurse can work more than 5 shifts in the 7-day week |

## Algorithm Pipeline
1. **Node Consistency** – remove on-leave nurses from domains
2. **AC-3** – enforce arc consistency to prune domains before search
3. **Backtracking Search** – with MRV heuristic and forward checking

In [ ]:
# Part 2 – Hospital Shift Scheduler
# ARI711S Artificial Intelligence – Group Project 2026

from collections import deque

DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
SHIFTS = ["Morning", "Afternoon", "Night"]
ALL_VARIABLES = [f"{day}_{shift}" for day in DAYS for shift in SHIFTS]
MAX_SHIFTS = 5


class Shift_AI_Solver:

    def __init__(self, nurses, leave_days):
        self.nurses = nurses
        self.leave_days = leave_days
        self.variables = ALL_VARIABLES[:]

        # build domain for each shift slot
        self.domains = {}
        for var in self.variables:
            day = var.rsplit("_", 1)[0]
            self.domains[var] = set()
            for nurse in nurses:
                if day not in leave_days.get(nurse, set()):
                    self.domains[var].add(nurse)

        # binary constraint arcs: Night -> next Morning (both directions)
        self.arcs = []
        for i in range(len(DAYS) - 1):
            night = f"{DAYS[i]}_Night"
            next_morning = f"{DAYS[i+1]}_Morning"
            self.arcs.append((night, next_morning))
            self.arcs.append((next_morning, night))

    def enforce_node_consistency(self):
        """Remove nurses on leave from each shift's domain."""
        for var in self.variables:
            day = var.rsplit("_", 1)[0]
            to_remove = {n for n in self.domains[var] if day in self.leave_days.get(n, set())}
            self.domains[var] -= to_remove

    def revise(self, x, y):
        """Make x arc-consistent with y. Returns True if domain(x) was reduced."""
        if (x, y) not in self.arcs:
            return False
        to_remove = set()
        for nurse in self.domains[x]:
            if not any(other != nurse for other in self.domains[y]):
                to_remove.add(nurse)
        if to_remove:
            self.domains[x] -= to_remove
            return True
        return False

    def ac3(self):
        """AC-3 algorithm. Returns False if any domain becomes empty."""
        queue = deque(self.arcs)
        while queue:
            x, y = queue.popleft()
            if self.revise(x, y):
                if len(self.domains[x]) == 0:
                    return False
                for (a, b) in self.arcs:
                    if b == x and a != y:
                        queue.append((a, x))
        return True

    def select_unassigned_variable(self, assignment):
        """MRV heuristic: pick shift with fewest nurses in domain."""
        unassigned = [v for v in self.variables if v not in assignment]
        best = unassigned[0]
        for v in unassigned:
            if len(self.domains[v]) < len(self.domains[best]):
                best = v
        return best

    def is_valid(self, var, nurse, assignment):
        """Check binary rest constraint and 5-shift limit."""
        day = var.rsplit("_", 1)[0]
        shift_type = var.rsplit("_", 1)[1]
        day_index = DAYS.index(day)

        if shift_type == "Night" and day_index < len(DAYS) - 1:
            if assignment.get(f"{DAYS[day_index + 1]}_Morning") == nurse:
                return False
        if shift_type == "Morning" and day_index > 0:
            if assignment.get(f"{DAYS[day_index - 1]}_Night") == nurse:
                return False

        if sum(1 for n in assignment.values() if n == nurse) >= MAX_SHIFTS:
            return False
        return True

    def backtrack(self, assignment):
        """Recursive backtracking search."""
        if len(assignment) == len(self.variables):
            return assignment
        var = self.select_unassigned_variable(assignment)
        for nurse in self.domains[var]:
            if self.is_valid(var, nurse, assignment):
                assignment[var] = nurse
                result = self.backtrack(assignment)
                if result is not None:
                    return result
                del assignment[var]
        return None

    def solve(self):
        """Run the full CSP pipeline."""
        print("Enforcing node consistency...")
        self.enforce_node_consistency()
        print("Running AC-3 algorithm...")
        if not self.ac3():
            print("No solution found.")
            return None
        print("Starting backtracking search...")
        return self.backtrack({})


print("Part 2 code loaded successfully.")

## Loading Staff Data and Running the Scheduler

In [ ]:
def load_staff(filename):
    nurses = []
    leave_days = {}
    with open(filename, "r") as f:
        for line in f:
            line = line.strip()
            if line == "" or line.startswith("#"):
                continue
            parts = [p.strip() for p in line.split(",")]
            name = parts[0]
            leave = {p for p in parts[1:] if p != ""}
            nurses.append(name)
            leave_days[name] = leave
    return nurses, leave_days


def print_schedule(schedule, leave_days):
    if schedule is None:
        print("Could not find a valid schedule.")
        return
    totals = {}
    for day in DAYS:
        print(f"\n{day.upper()}:")
        for shift in SHIFTS:
            var = f"{day}_{shift}"
            nurse = schedule[var]
            totals[nurse] = totals.get(nurse, 0) + 1
            off_list = [n for n, days in leave_days.items() if day in days]
            note = f"  (Note: {', '.join(off_list)} is Off)" if off_list else ""
            print(f"  {shift}: {nurse}{note}")
    print("\nSchedule Totals:")
    for nurse in sorted(totals):
        print(f"  - {nurse}: {totals[nurse]} shifts")
    print(f"\nStatus: Done! All {len(schedule)} shifts assigned. Constraints satisfied.")

In [ ]:
# Test with small staff file
print("=== SMALL DATASET ===")
nurses, leave_days = load_staff("part2/staff_small.txt")
print(f"Data loaded: {len(nurses)} staff members available.")
solver = Shift_AI_Solver(nurses, leave_days)
schedule = solver.solve()
print_schedule(schedule, leave_days)

In [ ]:
# Test with medium staff file
print("=== MEDIUM DATASET ===")
nurses, leave_days = load_staff("part2/staff_medium.txt")
print(f"Data loaded: {len(nurses)} staff members available.")
solver = Shift_AI_Solver(nurses, leave_days)
schedule = solver.solve()
print_schedule(schedule, leave_days)

In [ ]:
# Test with complex staff file
print("=== COMPLEX DATASET ===")
nurses, leave_days = load_staff("part2/staff_complex.txt")
print(f"Data loaded: {len(nurses)} staff members available.")
solver = Shift_AI_Solver(nurses, leave_days)
schedule = solver.solve()
print_schedule(schedule, leave_days)

---
# Part 3: Machine Learning – Traffic Sign Recognition
<a id='part3'></a>

## Background

We train a **Convolutional Neural Network (CNN)** to recognise 43 categories of German traffic signs using the GTSRB dataset, which contains over 50,000 images.

## Dataset

The GTSRB dataset is organised into 43 subfolders (named `0` to `42`), each containing JPEG images of one traffic sign category. All images are resized to **30×30 pixels** before training.

## CNN Architecture

| Layer | Details |
|---|---|
| Conv2D | 32 filters, 3×3, ReLU |
| MaxPooling2D | 2×2 |
| Conv2D | 64 filters, 3×3, ReLU |
| MaxPooling2D | 2×2 |
| Conv2D | 128 filters, 3×3, ReLU |
| Flatten | – |
| Dense | 256 neurons, ReLU |
| Dropout | 50% |
| Dense (output) | 43 neurons, Softmax |

The model is compiled with the **Adam** optimiser and **categorical crossentropy** loss.

In [ ]:
# Part 3 – Traffic Sign Recognition
# ARI711S Artificial Intelligence – Group Project 2026
# Install dependencies first if needed:
# pip install tensorflow opencv-python-headless scikit-learn numpy

import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = 30
NUM_CATEGORIES = 43
EPOCHS = 10
TEST_SPLIT = 0.2

print(f"TensorFlow version: {tf.__version__}")
print("Libraries loaded successfully.")

In [ ]:
def load_data(data_folder):
    """
    Loads all images from the GTSRB dataset.
    Folder should have 43 subfolders named 0 to 42.
    Returns numpy arrays of images and labels.
    """
    images = []
    labels = []

    for category in range(NUM_CATEGORIES):
        category_folder = os.path.join(data_folder, str(category))
        if not os.path.isdir(category_folder):
            print(f"Warning: folder {category_folder} not found, skipping")
            continue

        for filename in os.listdir(category_folder):
            if not filename.lower().endswith((".jpg", ".jpeg", ".png", ".ppm")):
                continue
            filepath = os.path.join(category_folder, filename)
            img = cv2.imread(filepath)
            if img is None:
                continue
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            images.append(img)
            labels.append(category)

    images = np.array(images, dtype="float32") / 255.0  # normalize to 0-1
    labels = np.array(labels)
    return images, labels


def build_model():
    """Builds and compiles the CNN model."""
    model = keras.Sequential([
        layers.Conv2D(32, (3, 3), activation="relu", input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(128, (3, 3), activation="relu"),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(NUM_CATEGORIES, activation="softmax")
    ])
    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


print("Functions defined. Ready to load data.")

In [ ]:
# Load the GTSRB dataset
# Make sure the gtsrb/ folder is in the same directory as this notebook
print("Loading data...")
images, labels = load_data("gtsrb")
print(f"Data loaded. Total images: {len(images)}, Categories: {NUM_CATEGORIES}")
print(f"Image shape: {images[0].shape}")

In [ ]:
# Split into training and testing sets (80% train, 20% test)
x_train, x_test, y_train, y_test = train_test_split(
    images, labels, test_size=TEST_SPLIT, random_state=42
)

# Convert labels to one-hot encoding
y_train_cat = keras.utils.to_categorical(y_train, NUM_CATEGORIES)
y_test_cat  = keras.utils.to_categorical(y_test,  NUM_CATEGORIES)

print(f"Training samples: {len(x_train)}")
print(f"Testing samples:  {len(x_test)}")

In [ ]:
# Build and train the CNN model
print("Training model...")
model = build_model()
model.summary()

model.fit(
    x_train, y_train_cat,
    epochs=EPOCHS,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# Evaluate the model on the test set
print("Evaluating model")
loss, accuracy = model.evaluate(x_test, y_test_cat, verbose=0)
print(f"Model accuracy: {accuracy:.4f}")

# Save the model
model.save("model.h5")
print("Model saved to model.h5")

---

## Summary

| Part | Topic | Algorithm | Result |
|---|---|---|---|
| 1 | Flight Connections | Breadth-First Search (BFS) | Finds shortest path between any two cities |
| 2 | Hospital Scheduling | CSP: Node Consistency + AC-3 + Backtracking + MRV | Valid 21-shift weekly schedule |
| 3 | Traffic Sign Recognition | Convolutional Neural Network (CNN) | ~95% classification accuracy |

---
*ARI711S Artificial Intelligence – Group Project 2026 – AI Project Ninjas*